# 02 - COVID-19 Data Cleaning

Here, we will analyze and correct dtypes, missing values, inconsistencies and duplicates.

## Import Libraries

In [15]:
import os
from pathlib import Path

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
os.chdir(project_root)

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import the data loader module
from src.data_loader import load_csv, get_basic_info
from src.visualization import set_style

# Set visualization style
set_style()

print('Libraries imported successfully!')

Libraries imported successfully!


## Load Raw Data

In [51]:
df = load_csv("data/raw/compact.csv")
df.head()

Data loaded successfully. Shape: (558807, 61)


,country,date,total_cases,new_cases,new_cases_smoothed,total_cases_per_million,new_cases_per_million,new_cases_smoothed_per_million,total_deaths,new_deaths,...,population,population_density,median_age,life_expectancy,gdp_per_capita,extreme_poverty,diabetes_prevalence,handwashing_facilities,hospital_beds_per_thousand,human_development_index
0,Afghanistan,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,40578847.0,62.215549,16.752001,65.616997,1516.273315,NaN,10.9,51.938343,0.39,NaN
1,Afghanistan,2020-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,40578847.0,62.215549,16.752001,65.616997,1516.273315,NaN,10.9,51.938343,0.39,NaN
2,Afghanistan,2020-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,40578847.0,62.215549,16.752001,65.616997,1516.273315,NaN,10.9,51.938343,0.39,NaN
3,Afghanistan,2020-01-04,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,...,40578847.0,62.215549,16.752001,65.616997,1516.273315,NaN,10.9,51.938343,0.39,NaN
4,Afghanistan,2020-01-05,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,...,40578847.0,62.215549,16.752001,65.616997,1516.273315,NaN,10.9,51.938343,0.39,NaN


## Change dtypes

In [57]:
df["date"] = pd.to_datetime(df["date"])
df.dtypes

country                               object
date                          datetime64[ns]
total_cases                          float64
new_cases                            float64
new_cases_smoothed                   float64
                                   ...      
extreme_poverty                      float64
diabetes_prevalence                  float64
handwashing_facilities               float64
hospital_beds_per_thousand           float64
human_development_index              float64
Length: 61, dtype: object

## Missing Values

In [58]:
missing = (df.isna().sum() / len(df)).sort_values(ascending=False)
missing

human_development_index                    1.000000
weekly_icu_admissions                      0.980328
weekly_icu_admissions_per_million          0.980328
excess_mortality_cumulative_per_million    0.975457
excess_mortality_cumulative_absolute       0.975396
                                             ...   
total_cases_per_million                    0.022816
total_deaths                               0.022816
total_deaths_per_million                   0.022816
date                                       0.000000
country                                    0.000000
Length: 61, dtype: float64

After looking at all missing values we get:

* Human Development Index: 100% missing -> drop
* Cases and death: low missing values
* Excess mortality: +97% -> drop
* ICU and Hospital patients: ~93% -> use with available (can be interesting)
* Stringency index: 63% -> use with available data
* Reproduction rate: 66% -> use with available
* Tests: ~85% -> use with available (robust testing such as EU, USA, China, etc.)
* Vaccination: ~85% -> use only with available
* Handwashing, hospital beds, extreme poverty, gdp: useful for analysis and with a low rate of missing (~ <50%)

**Practical Strategy**
1. Mandatory variables: cases, deaths, country, continent, population
2. Optional variables (for sub-analysis with complete data): vaccination, stringency and reproduction rate, hospital and icu, tests
3. Context or socioeconomical variables: gdp, median age, life expectancy, hospital beds, handwashing.
4. Drop: excess mortality, weekly ICU/hospital admissions, total boosters

**Imputation tips and cleaning**

* Socioeconomic variables: impute with the median by continent.
* Vaccination variables and tests: not impute with 0, use only countries with data in each analysis
* Rolling averages (new_cases_smoothed): use them for temporal plots.

### Dropping columns

In [59]:
cols_to_drop = [
    "human_development_index",
    "weekly_icu_admissions",
    "weekly_icu_admissions_per_million",
    "excess_mortality",
    "excess_mortality_cumulative",
    "excess_mortality_cumulative_absolute",
    "excess_mortality_cumulative_per_million",
    "total_boosters",
    "total_boosters_per_hundred"
]

df = df.drop(columns=cols_to_drop, axis=1)

### Optional Variables

In [60]:
vacc_df = df.dropna(subset=["people_vaccinated"]).copy()

### Socioeconomic Variables

In [61]:
for col in ["gdp_per_capita","median_age","life_expectancy","hospital_beds_per_thousand","handwashing_facilities","extreme_poverty"]:
    df[col] = df.groupby("continent")[col].transform(lambda x: x.fillna(x.median()))

In [62]:
(df.isna().sum() / len(df)).sort_values(ascending=False)

weekly_hosp_admissions_per_million            0.956162
weekly_hosp_admissions                        0.956162
icu_patients_per_million                      0.930001
icu_patients                                  0.930001
hosp_patients                                 0.927245
hosp_patients_per_million                     0.927245
new_vaccinations                              0.876745
new_tests_per_thousand                        0.865064
new_tests                                     0.865064
people_fully_vaccinated                       0.862187
people_fully_vaccinated_per_hundred           0.862187
people_vaccinated                             0.858608
people_vaccinated_per_hundred                 0.858608
total_tests                                   0.857935
total_tests_per_thousand                      0.857935
total_vaccinations                            0.851072
total_vaccinations_per_hundred                0.851072
tests_per_case                                0.819884
positive_r

## Data Cleaning Steps

## Save Processed Data